# EgoBrain — EEG Temporal Teacher Diagnostics

**Purpose:** independently characterize the 128-D EEG modality under the frozen subject-independent action-recognition protocol. This notebook is diagnostic-first: it compares pooled and temporal teachers, measures temporal-order sensitivity and importance concentration, and writes a reusable teacher/cache package.

### Scientific interpretation of the frozen run
The latest completed run selected a temporal Transformer teacher with test Macro-F1 ≈ 0.03888 for a 27-class task (uniform chance ≈ 0.03704). The temporal-shuffle Macro-F1 drop was ≈ 0.00088 and normalized importance entropy ≈ 0.99978, so the evidence should be described as weak neural action information with limited temporal concentration, not as a strong neural temporal prior.


## Inputs — only the existing frozen inputs are required

### Required Kaggle dataset/input mounts
Use the same datasets already attached to the earlier Phase-3 notebook:

```text
1. Phase-1 output
/kaggle/input/datasets/keerthikumarkj/iris-egobrain-actual-phase-1-output

2. EEG feature tensors
/kaggle/input/datasets/keerthikumarkj/iris-eeg-features/eeg_features

3. Phase-2 actual outputs
/kaggle/input/datasets/keerthikumarkj/iris-phase-2-actual-outputs
```

EEG-teacher stage actually reads only:

```text
Phase 1:
  all_subjects_synchronized_labeled_index.csv

Phase 2:
  sequence_index_with_split.csv

EEG:
  eeg_P0001.pt ... eeg_P0040.pt
  (or the equivalent subject feature filenames already present)
```

### NOT required for EEG-teacher stage

```text
✗ raw EDF files
✗ Hugging Face download
✗ raw video / MP4
✗ 512-D visual feature tensors
✗ Phase-2 visual Transformer checkpoint
```

The Phase-2 output mount is needed only because the frozen `sequence_index_with_split.csv` lives there. The visual tensors/checkpoint are deliberately not loaded.

In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION / REPRODUCIBILITY LOCK
# ============================================================
from pathlib import Path
import os, json, math, time, random, shutil, hashlib, warnings
from collections import OrderedDict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 20260928
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

SUBJECTS = [f"P{i:04d}" for i in range(1, 41)]
EEG_DIM = 128
SEQ_LEN = 8
TRAIN_EXPECTED = 31147
VAL_EXPECTED = 38404
TEST_EXPECTED = 46178
EXPECTED_NUM_CLASSES = 27
NUM_WORKERS = 0

# Teacher model budget.
BATCH_SIZE = 256 if DEVICE.type == "cuda" else 128
EPOCHS = 10
PATIENCE = 3
LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
TOKEN_LOSS_WEIGHT = 0.50
IMPORTANCE_TEMPERATURE = 0.25

# Diagnostics / selection are predeclared before test evaluation.
CANDIDATE_NAMES = [
    "pooled_logistic",
    "mean_mlp",
    "temporal_conv",
    "temporal_transformer",
]
TEMPORAL_TEACHER_CANDIDATES = ["temporal_conv", "temporal_transformer"]
CHANCE_F1 = 1.0 / EXPECTED_NUM_CLASSES
VAL_MARGIN = 0.003
TEST_MARGIN = 0.005
TOKEN_MARGIN = 0.003
SHUFFLE_DROP_MIN = 0.002
IMPORTANCE_ENTROPY_MAX = 0.995
ADJACENT_COSINE_MAX = 0.995

# Save targets in float16 only after teacher evaluation.
CACHE_FP16 = True

PHASE1_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-egobrain-actual-phase-1-output"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase-1-output/egobrain_phase1"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase-1-output"),
]
EEG_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-eeg-features/eeg_features"),
    Path("/kaggle/input/datasets/keerthikumarkj/eeg_features"),
]
PHASE2_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-phase-2-actual-outputs"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase2-actual-outputs"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain_phase2_artifacts"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase2"),
]

OUT_ROOT = Path("/kaggle/working/egobrain_phase3_1_eeg_teacher")
MODEL_ROOT = OUT_ROOT / "models"
CACHE_ROOT = OUT_ROOT / "teacher_cache"
REPORT_ROOT = OUT_ROOT / "reports"
FIG_ROOT = OUT_ROOT / "figures"
for p in [OUT_ROOT, MODEL_ROOT, CACHE_ROOT, REPORT_ROOT, FIG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("SEQ_LEN:", SEQ_LEN)
print("Expected classes:", EXPECTED_NUM_CLASSES)
print("Chance macro-F1:", CHANCE_F1)


In [ ]:
# ============================================================
# CELL 2 — RESOLVE ONLY THE THREE REQUIRED INPUT MOUNTS
# ============================================================
def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

def find_file(root, names):
    for name in names:
        direct = root / name
        if direct.exists():
            return direct
    for name in names:
        hits = sorted(root.rglob(name))
        if hits:
            return hits[0]
    return None

PHASE1_ROOT = first_existing(PHASE1_ROOT_CANDIDATES)
EEG_ROOT = first_existing(EEG_ROOT_CANDIDATES)
PHASE2_ROOT = first_existing(PHASE2_ROOT_CANDIDATES)

if PHASE1_ROOT is None:
    raise FileNotFoundError("Phase-1 output mount not found.")
if EEG_ROOT is None:
    raise FileNotFoundError("EEG feature mount not found.")
if PHASE2_ROOT is None:
    raise FileNotFoundError("Phase-2 output mount not found.")

LABELED_INDEX = find_file(PHASE1_ROOT, ["all_subjects_synchronized_labeled_index.csv"])
PHASE2_SPLIT = find_file(PHASE2_ROOT, ["sequence_index_with_split.csv"])

if LABELED_INDEX is None:
    raise FileNotFoundError("Frozen Phase-1 all_subjects_synchronized_labeled_index.csv not found.")
if PHASE2_SPLIT is None:
    raise FileNotFoundError("Frozen Phase-2 sequence_index_with_split.csv not found.")

print("PHASE1_ROOT:", PHASE1_ROOT)
print("EEG_ROOT:", EEG_ROOT)
print("PHASE2_ROOT:", PHASE2_ROOT)
print("LABELED_INDEX:", LABELED_INDEX)
print("PHASE2_SPLIT:", PHASE2_SPLIT)
print("No visual tensor/checkpoint will be loaded in Phase 3.1.")


## 1. Frozen Phase-1 ↔ Phase-2 sequence integrity

The teacher must consume **exactly the same 8 consecutive 1-second EEG intervals** that correspond to each frozen Phase-2 sequence.

This cell checks, for every sequence row:

- the frozen split counts;
- subject disjointness;
- `sync_end - sync_start == 8`;
- the corresponding Phase-1 rows exist;
- EEG feature indices are consecutive;
- visual feature indices are consecutive;
- timestamps span exactly 8 seconds;
- the eight target labels remain identical.

No labels are reconstructed here; the frozen Phase-1/Phase-2 metadata is authoritative.

In [ ]:
# ============================================================
# CELL 3 — FAST FROZEN EEG SEQUENCE RECONSTRUCTION
#
# IMPORTANT:
# The full row-wise EEG sequence integrity audit has already
# been completed successfully:
#   train = 31,147 / 0 problematic
#   val   = 38,404 / 0 problematic
#   test  = 46,178 / 0 problematic
#
# DO NOT rerun that expensive audit here.
#
# This cell only reconstructs the exact audited metadata
# vectorized from the frozen Phase-1 / Phase-2 indexes.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("FAST FROZEN EEG SEQUENCE RECONSTRUCTION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load frozen indexes
# ------------------------------------------------------------

phase1_index = pd.read_csv(LABELED_INDEX)
seq_df = pd.read_csv(PHASE2_SPLIT)

print("Phase-1 rows:", f"{len(phase1_index):,}")
print("Phase-2 sequence rows:", f"{len(seq_df):,}")

# ------------------------------------------------------------
# 2. Keep only canonical train / val / test sequences
# ------------------------------------------------------------

seq_df = (
    seq_df[
        seq_df["split"].isin(["train", "val", "test"])
    ]
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Frozen 27-class label space
# ------------------------------------------------------------

NUM_CLASSES = int(seq_df["label_id"].max()) + 1

if NUM_CLASSES != EXPECTED_NUM_CLASSES:
    raise RuntimeError(
        f"Expected {EXPECTED_NUM_CLASSES} classes, "
        f"found {NUM_CLASSES}"
    )

if sorted(seq_df["label_id"].unique().tolist()) != list(
    range(EXPECTED_NUM_CLASSES)
):
    raise RuntimeError(
        "Frozen label IDs are not contiguous 0..26."
    )

label_map = (
    seq_df[
        ["label_id", "action_label"]
    ]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

label_to_id = dict(
    zip(
        label_map["action_label"].astype(str),
        label_map["label_id"].astype(int),
    )
)

print(f"PASS frozen 27-class label space")

# ------------------------------------------------------------
# 4. Reconstruct the EEG start index
#
# For every Phase-2 sequence:
#
#     sync_start
#          ↓
#     Phase-1 row at (subject_id, sync_start)
#          ↓
#     eeg_feature_idx
#
# The previous full audit already established that the next
# seven synchronized rows are consecutive EEG rows.
#
# Therefore:
#
#     eeg_start = eeg_feature_idx at sync_start
#     eeg_end   = eeg_start + 7
# ------------------------------------------------------------

p1_start_map = (
    phase1_index[
        [
            "subject_id",
            "sync_index",
            "eeg_feature_idx",
            "action_label",
            "start_sec",
            "end_sec",
        ]
    ]
    .rename(
        columns={
            "sync_index": "sync_start",
            "eeg_feature_idx": "eeg_start",
            "action_label": "p1_action_label",
            "start_sec": "p1_start_sec",
            "end_sec": "p1_first_window_end_sec",
        }
    )
)

# Required uniqueness for lookup.
if p1_start_map[
    ["subject_id", "sync_start"]
].duplicated().any():

    raise RuntimeError(
        "Phase-1 start lookup is not unique."
    )

# ------------------------------------------------------------
# 5. Vectorized join
# ------------------------------------------------------------

seq_aug = seq_df.merge(
    p1_start_map,
    on=["subject_id", "sync_start"],
    how="left",
    validate="many_to_one",
)

# ------------------------------------------------------------
# 6. Make sure every frozen sequence resolved
# ------------------------------------------------------------

missing_start = seq_aug["eeg_start"].isna()

if missing_start.any():

    bad = seq_aug.loc[
        missing_start,
        ["subject_id", "sync_start", "sync_end"]
    ].head(10)

    raise RuntimeError(
        "Unable to reconstruct eeg_start for "
        f"{missing_start.sum():,} sequences.\n"
        f"{bad.to_string(index=False)}"
    )

# ------------------------------------------------------------
# 7. Cheap vectorized label sanity check
# ------------------------------------------------------------

label_mismatch = (
    seq_aug["action_label"].astype(str)
    !=
    seq_aug["p1_action_label"].astype(str)
)

if label_mismatch.any():

    bad = seq_aug.loc[
        label_mismatch,
        [
            "subject_id",
            "sync_start",
            "action_label",
            "p1_action_label",
        ]
    ].head(10)

    raise RuntimeError(
        "Phase-1 / Phase-2 action-label mismatch detected.\n"
        f"{bad.to_string(index=False)}"
    )

print("PASS Phase-1 -> Phase-2 EEG label mapping")

# ------------------------------------------------------------
# 8. Construct exact 8-token EEG span
# ------------------------------------------------------------

seq_aug["eeg_start"] = (
    seq_aug["eeg_start"]
    .astype(np.int64)
)

seq_aug["eeg_end"] = (
    seq_aug["eeg_start"]
    + SEQ_LEN
    - 1
)

seq_aug["duration_sec"] = float(SEQ_LEN)

# ------------------------------------------------------------
# 9. Remove temporary lookup columns
# ------------------------------------------------------------

seq_aug = seq_aug.drop(
    columns=[
        "p1_action_label",
        "p1_start_sec",
        "p1_first_window_end_sec",
    ]
)

# ------------------------------------------------------------
# 10. Recreate BOTH naming conventions
#
# Existing downstream code may use either:
#
#     train_seq / val_seq / test_seq
#
# or:
#
#     TRAIN_SEQ / VAL_SEQ / TEST_SEQ
#
# Make both identical and authoritative.
# ------------------------------------------------------------

train_seq = (
    seq_aug[
        seq_aug["split"] == "train"
    ]
    .reset_index(drop=True)
)

val_seq = (
    seq_aug[
        seq_aug["split"] == "val"
    ]
    .reset_index(drop=True)
)

test_seq = (
    seq_aug[
        seq_aug["split"] == "test"
    ]
    .reset_index(drop=True)
)

TRAIN_SEQ = train_seq.copy()
VAL_SEQ = val_seq.copy()
TEST_SEQ = test_seq.copy()

# ------------------------------------------------------------
# 11. Frozen count gate
# ------------------------------------------------------------

actual_counts = {
    "train": len(TRAIN_SEQ),
    "val": len(VAL_SEQ),
    "test": len(TEST_SEQ),
}

expected_counts = {
    "train": TRAIN_EXPECTED,
    "val": VAL_EXPECTED,
    "test": TEST_EXPECTED,
}

print("Frozen counts:", actual_counts)

if actual_counts != expected_counts:
    raise RuntimeError(
        "Frozen sequence counts changed.\n"
        f"Expected: {expected_counts}\n"
        f"Actual:   {actual_counts}"
    )

# ------------------------------------------------------------
# 12. Subject split gate
# ------------------------------------------------------------

train_subjects = sorted(
    TRAIN_SEQ["subject_id"].unique().tolist()
)

val_subjects = sorted(
    VAL_SEQ["subject_id"].unique().tolist()
)

test_subjects = sorted(
    TEST_SEQ["subject_id"].unique().tolist()
)

if (
    len(train_subjects),
    len(val_subjects),
    len(test_subjects),
) != (24, 8, 8):

    raise RuntimeError(
        "Frozen subject split is not 24/8/8."
    )

if set(train_subjects) & set(val_subjects):
    raise RuntimeError("Train/validation subject overlap.")

if set(train_subjects) & set(test_subjects):
    raise RuntimeError("Train/test subject overlap.")

if set(val_subjects) & set(test_subjects):
    raise RuntimeError("Validation/test subject overlap.")

print("PASS subject-disjoint frozen split")

# ------------------------------------------------------------
# 13. Final metadata sanity
# ------------------------------------------------------------

required_eeg_cols = {
    "subject_id",
    "sync_start",
    "sync_end",
    "eeg_start",
    "eeg_end",
    "action_label",
    "label_id",
    "split",
}

missing_cols = sorted(
    required_eeg_cols - set(TRAIN_SEQ.columns)
)

if missing_cols:
    raise RuntimeError(
        f"Required EEG metadata missing: {missing_cols}"
    )

# ------------------------------------------------------------
# 14. Report the reconstructed range
# ------------------------------------------------------------

all_seq = pd.concat(
    [TRAIN_SEQ, VAL_SEQ, TEST_SEQ],
    ignore_index=True,
)

print(
    "EEG start range:",
    int(all_seq["eeg_start"].min()),
    "->",
    int(all_seq["eeg_start"].max()),
)

print(
    "EEG end range:",
    int(all_seq["eeg_end"].min()),
    "->",
    int(all_seq["eeg_end"].max()),
)

print(
    "Example reconstructed sequence:"
)

print(
    all_seq[
        [
            "subject_id",
            "sync_start",
            "sync_end",
            "eeg_start",
            "eeg_end",
            "action_label",
            "split",
        ]
    ]
    .head(5)
    .to_string(index=False)
)

print("=" * 80)
print("PASS FAST EEG SEQUENCE RECONSTRUCTION")
print("NO EXPENSIVE ROW-WISE AUDIT WAS RERUN")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 4 — LOAD EEG FEATURE TENSORS + RANGE / FINITENESS AUDIT
# ============================================================
def safe_torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def find_subject_file(subject):
    candidates = [EEG_ROOT / f"eeg_{subject}.pt", EEG_ROOT / f"features_{subject}.pt"]
    for p in candidates:
        if p.exists():
            return p
    hits = sorted(EEG_ROOT.glob(f"*{subject}*.pt"))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"No EEG feature file found for {subject}")

eeg_cache = {}
inventory_rows = []

for subject in SUBJECTS:
    path = find_subject_file(subject)
    obj = safe_torch_load(path)
    meta = obj if isinstance(obj, dict) else {}
    feat = obj["features"] if isinstance(obj, dict) and "features" in obj else obj
    feat = torch.as_tensor(feat).float().contiguous()

    if feat.ndim != 2 or int(feat.shape[1]) != EEG_DIM:
        raise RuntimeError(f"{subject}: expected [N,{EEG_DIM}], got {tuple(feat.shape)}")
    if not torch.isfinite(feat).all():
        raise RuntimeError(f"{subject}: EEG feature tensor contains NaN/Inf")

    # The frozen Phase-1 map tells us the maximum EEG row used by Phase-2.
    max_needed = int(pd.concat([train_seq, val_seq, test_seq]).query("subject_id == @subject")["eeg_start"].max()) + SEQ_LEN
    if max_needed > int(feat.shape[0]):
        raise RuntimeError(f"{subject}: EEG tensor too short. Need {max_needed}, have {feat.shape[0]}")

    eeg_cache[subject] = feat
    inventory_rows.append({
        "subject_id": subject,
        "rows": int(feat.shape[0]),
        "feature_dim": int(feat.shape[1]),
        "path": str(path),
        "sampling_rate_hz": meta.get("sampling_rate_hz", None) if isinstance(meta, dict) else None,
    })

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(REPORT_ROOT / "eeg_feature_inventory.csv", index=False)
display(inventory)

# Full-row coverage audit for actual sequence starts.
for split_name, table in [("train", train_seq), ("val", val_seq), ("test", test_seq)]:
    for r in table.itertuples(index=False):
        x = eeg_cache[r.subject_id][int(r.eeg_start):int(r.eeg_start)+SEQ_LEN]
        if tuple(x.shape) != (SEQ_LEN, EEG_DIM):
            raise RuntimeError(f"{split_name}: invalid EEG window {r.subject_id}:{r.eeg_start}")

print("PASS all 40 EEG feature tensors + all frozen EEG windows are addressable")


## 2. Train-only normalization and EEG feature-health diagnostics

Normalization is computed **only from the training split**. Validation and test are never used to calculate feature statistics.

The diagnostics are designed to answer the exact failure observed in the previous Phase-3 run:

> Are consecutive EEG embeddings almost identical, making the learned temporal teacher effectively blind to time?

In [ ]:
# ============================================================
# CELL 5 — TRAIN-ONLY EEG NORMALIZATION
# ============================================================
sum_x = torch.zeros(EEG_DIM, dtype=torch.float64)
sum_x2 = torch.zeros(EEG_DIM, dtype=torch.float64)
count = 0

for r in train_seq.itertuples(index=False):
    x = eeg_cache[r.subject_id][int(r.eeg_start):int(r.eeg_start)+SEQ_LEN].double()
    sum_x += x.sum(0)
    sum_x2 += (x * x).sum(0)
    count += int(x.shape[0])

train_eeg_mean = (sum_x / count).float()
train_eeg_var = (sum_x2 / count - (sum_x / count) ** 2).clamp_min(1e-8).float()
train_eeg_std = train_eeg_var.sqrt().clamp_min(1e-4)

torch.save(train_eeg_mean, OUT_ROOT / "train_eeg_mean.pt")
torch.save(train_eeg_std, OUT_ROOT / "train_eeg_std.pt")

norm_report = {
    "tokens_used": int(count),
    "feature_dim": EEG_DIM,
    "mean_abs": float(train_eeg_mean.abs().mean()),
    "std_mean": float(train_eeg_std.mean()),
    "std_min": float(train_eeg_std.min()),
    "std_max": float(train_eeg_std.max()),
}
with open(REPORT_ROOT / "train_only_eeg_normalization.json", "w") as f:
    json.dump(norm_report, f, indent=2)
print(json.dumps(norm_report, indent=2))


In [ ]:
# ============================================================
# CELL 6 — FEATURE HEALTH / TEMPORAL COLLAPSE / CLASS SEPARABILITY
# ============================================================
def sample_windows(table, n, seed):
    rng = np.random.default_rng(seed)
    n = min(int(n), len(table))
    idx = rng.choice(len(table), size=n, replace=False)
    xs, ys, ss = [], [], []
    for i in idx:
        r = table.iloc[int(i)]
        x = eeg_cache[r.subject_id][int(r.eeg_start):int(r.eeg_start)+SEQ_LEN]
        xs.append((x - train_eeg_mean) / train_eeg_std)
        ys.append(int(r.label_id))
        ss.append(r.subject_id)
    return torch.stack(xs).float(), np.asarray(ys), np.asarray(ss)

health_table = pd.concat([train_seq, val_seq], ignore_index=True)
health_x, health_y, health_s = sample_windows(health_table, 12000, SEED + 100)

feature_std = health_x.reshape(-1, EEG_DIM).std(0)
near_constant = feature_std < 1e-5

z = F.normalize(health_x, dim=-1)
adj_cos = (z[:, :-1] * z[:, 1:]).sum(-1)
temporal_diff = (health_x[:, 1:] - health_x[:, :-1]).norm(dim=-1)
relative_diff = temporal_diff / health_x[:, :-1].norm(dim=-1).clamp_min(1e-6)

# A simple class-separability diagnostic on sequence means.
pooled = health_x.mean(1).numpy()
centroids = {}
for c in np.unique(health_y):
    centroids[int(c)] = pooled[health_y == c].mean(0)
within = []
between = []
for i, c in enumerate(health_y):
    d = np.linalg.norm(pooled[i] - centroids[int(c)])
    within.append(d)
for c1 in list(centroids.keys()):
    for c2 in list(centroids.keys()):
        if c2 > c1:
            between.append(np.linalg.norm(centroids[c1] - centroids[c2]))

health_report = {
    "sampled_windows": int(len(health_x)),
    "near_constant_dimensions": int(near_constant.sum()),
    "adjacent_cosine_mean": float(adj_cos.mean()),
    "adjacent_cosine_p95": float(torch.quantile(adj_cos, 0.95)),
    "within_window_variance_mean": float(health_x.var(dim=1).mean()),
    "temporal_difference_norm_mean": float(temporal_diff.mean()),
    "relative_temporal_difference_mean": float(relative_diff.mean()),
    "class_pooled_within_distance_mean": float(np.mean(within)),
    "class_centroid_between_distance_mean": float(np.mean(between)),
    "between_to_within_ratio": float(np.mean(between) / max(np.mean(within), 1e-8)),
}
with open(REPORT_ROOT / "feature_health_diagnostics.json", "w") as f:
    json.dump(health_report, f, indent=2)
print(json.dumps(health_report, indent=2))


## 3. Candidate EEG teachers

The comparison is deliberate:

- **Pooled Logistic Regression**: asks whether simple summary statistics already separate actions.
- **Mean/Std MLP**: a nonlinear non-temporal baseline.
- **Temporal Conv**: local temporal dynamics.
- **Temporal Transformer V2**: the canonical research teacher.

The pooled baselines are **diagnostic baselines only**. The actual Phase-3.1 teacher must be a temporal model.

### Teacher V2 repair
The previous teacher had a free learned importance head. That head produced nearly uniform importance. V2 instead derives temporal importance from **per-token class confidence**:

`high-confidence token → lower token entropy → higher temporal importance`.

It also adds token-level supervision so every 1-second EEG token is forced to carry action-relevant information.

In [ ]:
# ============================================================
# CELL 7 — DATASET / MODEL DEFINITIONS
# ============================================================
CHANCE_F1 = 1.0 / NUM_CLASSES

class EEGSequenceDataset(Dataset):
    def __init__(self, table):
        self.df = table.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = eeg_cache[r.subject_id][int(r.eeg_start):int(r.eeg_start)+SEQ_LEN].float()
        x = (x - train_eeg_mean) / train_eeg_std
        return x, int(r.label_id), i

train_ds = EEGSequenceDataset(train_seq)
val_ds = EEGSequenceDataset(val_seq)
test_ds = EEGSequenceDataset(test_seq)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)
train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

def autocast_ctx():
    return torch.autocast("cuda", dtype=torch.float16) if DEVICE.type == "cuda" else nullcontext()

def metrics(y, p):
    return {
        "accuracy": float(accuracy_score(y, p)),
        "balanced_accuracy": float(balanced_accuracy_score(y, p)),
        "macro_f1": float(f1_score(y, p, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y, p, average="weighted", zero_division=0)),
    }

# Class weights are derived strictly from the frozen training split.
train_counts = np.bincount(train_seq.label_id.to_numpy(), minlength=NUM_CLASSES).astype(np.float64)
class_weight_np = train_counts.sum() / np.maximum(train_counts, 1.0)
class_weight_np = class_weight_np / class_weight_np.mean()
CLASS_WEIGHTS = torch.tensor(class_weight_np, dtype=torch.float32, device=DEVICE)

def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

class MeanStdMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(EEG_DIM * 2, 256), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(256, 128), nn.GELU(), nn.Linear(128, NUM_CLASSES)
        )
    def forward(self, x):
        pooled = torch.cat([x.mean(1), x.std(1)], dim=-1)
        return {"logits": self.net(pooled)}

class TemporalConvTeacher(nn.Module):
    def __init__(self):
        super().__init__()
        d = 128
        self.input_norm = nn.LayerNorm(EEG_DIM)
        self.proj = nn.Linear(EEG_DIM, d)
        self.conv = nn.Sequential(
            nn.Conv1d(d, d, 3, padding=1), nn.GELU(), nn.BatchNorm1d(d),
            nn.Conv1d(d, d, 3, padding=1), nn.GELU(), nn.BatchNorm1d(d),
        )
        self.token_classifier = nn.Linear(d, NUM_CLASSES)
        self.global_classifier = nn.Linear(d, NUM_CLASSES)
    def forward(self, x):
        z = self.proj(self.input_norm(x)).transpose(1, 2)
        z = self.conv(z).transpose(1, 2)
        token_logits = self.token_classifier(z)
        token_prob = torch.softmax(token_logits, dim=-1)
        entropy = -(token_prob.clamp_min(1e-8) * token_prob.clamp_min(1e-8).log()).sum(-1)
        importance_logits = -entropy / IMPORTANCE_TEMPERATURE
        importance = torch.softmax(importance_logits, dim=1)
        pooled = (z * importance.unsqueeze(-1)).sum(1)
        logits = self.global_classifier(pooled)
        return {
            "tokens": z,
            "token_logits": token_logits,
            "importance_logits": importance_logits,
            "importance": importance,
            "entropy": entropy,
            "logits": logits,
        }

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=SEQ_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)
    def forward(self, x):
        return x + self.pe[:x.shape[1]].unsqueeze(0).to(x.dtype)

class EEGTemporalTeacherV2(nn.Module):
    def __init__(self):
        super().__init__()
        d = 160
        self.input_norm = nn.LayerNorm(EEG_DIM)
        self.input_proj = nn.Linear(EEG_DIM, d)
        self.pos = SinusoidalPE(d)
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=4, dim_feedforward=320, dropout=0.10,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.norm = nn.LayerNorm(d)
        self.token_classifier = nn.Linear(d, NUM_CLASSES)
        self.global_classifier = nn.Linear(d, NUM_CLASSES)
    def forward(self, x):
        z = self.input_proj(self.input_norm(x))
        z = self.norm(self.encoder(self.pos(z)))
        token_logits = self.token_classifier(z)
        token_prob = torch.softmax(token_logits, dim=-1)
        entropy = -(token_prob.clamp_min(1e-8) * token_prob.clamp_min(1e-8).log()).sum(-1)
        importance_logits = -entropy / IMPORTANCE_TEMPERATURE
        importance = torch.softmax(importance_logits, dim=1)
        pooled = (z * importance.unsqueeze(-1)).sum(1)
        logits = self.global_classifier(pooled)
        return {
            "tokens": z,
            "token_logits": token_logits,
            "importance_logits": importance_logits,
            "importance": importance,
            "entropy": entropy,
            "logits": logits,
        }

def candidate_loss(name, out, y):
    global_ce = F.cross_entropy(out["logits"], y, weight=CLASS_WEIGHTS)
    if name in TEMPORAL_TEACHER_CANDIDATES:
        b, t, c = out["token_logits"].shape
        token_y = y[:, None].expand(b, t).reshape(-1)
        token_logits = out["token_logits"].reshape(-1, c)
        token_ce = F.cross_entropy(token_logits, token_y, weight=CLASS_WEIGHTS)
        return global_ce + TOKEN_LOSS_WEIGHT * token_ce
    return global_ce

def make_model(name):
    if name == "mean_mlp": return MeanStdMLP()
    if name == "temporal_conv": return TemporalConvTeacher()
    if name == "temporal_transformer": return EEGTemporalTeacherV2()
    raise ValueError(name)

def numpy_pooled(table):
    X, y = [], []
    for r in table.itertuples(index=False):
        x = eeg_cache[r.subject_id][int(r.eeg_start):int(r.eeg_start)+SEQ_LEN].float()
        x = (x - train_eeg_mean) / train_eeg_std
        X.append(torch.cat([x.mean(0), x.std(0)]).numpy())
        y.append(int(r.label_id))
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int64)

print("NUM_CLASSES:", NUM_CLASSES, "| SEQ_LEN:", SEQ_LEN, "| EEG_DIM:", EEG_DIM)


In [ ]:
# ============================================================
# CELL 8 — POOLED LOGISTIC BASELINE (DIAGNOSTIC ONLY)
# ============================================================
X_train, y_train = numpy_pooled(train_seq)
X_val, y_val = numpy_pooled(val_seq)
X_test, y_test = numpy_pooled(test_seq)

logreg = LogisticRegression(
    C=1.0,
    max_iter=300,
    class_weight="balanced",
    solver="lbfgs",
    multi_class="auto",
    random_state=SEED,
)
logreg.fit(X_train, y_train)

logreg_val = metrics(y_val, logreg.predict(X_val))
logreg_test = metrics(y_test, logreg.predict(X_test))

logreg_report = pd.DataFrame([
    {"split":"val", **logreg_val},
    {"split":"test", **logreg_test},
])
logreg_report.to_csv(REPORT_ROOT / "pooled_logistic_diagnostic.csv", index=False)
display(logreg_report)


In [ ]:
# ============================================================
# CELL 9 — TRAIN TEMPORAL / MLP CANDIDATES WITH VALIDATION-ONLY SELECTION
# ============================================================
def eval_model(model, loader, return_outputs=False):
    model.eval()
    ys, ps = [], []
    outputs = []
    with torch.no_grad():
        for xb, yb, idx in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with autocast_ctx():
                out = model(xb)
            ys.extend(yb.numpy().tolist())
            ps.extend(out["logits"].argmax(1).cpu().numpy().tolist())
            if return_outputs and "tokens" in out:
                outputs.append({
                    "idx": idx.numpy(),
                    "tokens": out["tokens"].float().cpu(),
                    "importance": out["importance"].float().cpu(),
                })
    m = metrics(np.asarray(ys), np.asarray(ps))
    return (m, outputs) if return_outputs else m

candidate_rows = []
candidate_states = {}
candidate_histories = {}

for name in ["mean_mlp", "temporal_conv", "temporal_transformer"]:
    print("\n" + "=" * 78)
    print("CANDIDATE:", name)
    print("=" * 78)

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    model = make_model(name).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_f1 = -np.inf
    best_state = None
    patience = 0
    history = []

    for epoch in range(EPOCHS):
        model.train()
        run_loss = 0.0
        seen = 0
        for xb, yb, _ in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast_ctx():
                out = model(xb)
                loss = candidate_loss(name, out, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            run_loss += float(loss.detach().cpu()) * len(yb)
            seen += len(yb)
        sched.step()

        vm = eval_model(model, val_loader)
        row = {
            "epoch": epoch + 1,
            "train_loss": run_loss / max(seen, 1),
            **{f"val_{k}": v for k, v in vm.items()},
            "lr": opt.param_groups[0]["lr"],
        }
        history.append(row)
        print(f"{name} | epoch {epoch+1:02d} | loss {row['train_loss']:.4f} | val F1 {vm['macro_f1']:.4f}")

        if vm["macro_f1"] > best_f1 + 1e-6:
            best_f1 = vm["macro_f1"]
            best_state = clone_state(model)
            patience = 0
        else:
            patience += 1
        if patience >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(f"No best state for {name}")

    model.load_state_dict(best_state)
    vm = eval_model(model, val_loader)
    candidate_states[name] = best_state
    candidate_histories[name] = history
    candidate_rows.append({
        "candidate": name,
        "candidate_type": "temporal" if name in TEMPORAL_TEACHER_CANDIDATES else "pooled",
        "val_macro_f1": vm["macro_f1"],
        "val_balanced_accuracy": vm["balanced_accuracy"],
        "parameters": int(sum(p.numel() for p in model.parameters())),
        "epochs_ran": len(history),
    })
    pd.DataFrame(history).to_csv(REPORT_ROOT / f"{name}_history.csv", index=False)

candidate_df = pd.DataFrame(candidate_rows)
candidate_df.to_csv(REPORT_ROOT / "candidate_validation_comparison.csv", index=False)
display(candidate_df.sort_values("val_macro_f1", ascending=False))

# The actual teacher can only come from a temporal candidate.
temporal_df = candidate_df[candidate_df.candidate.isin(TEMPORAL_TEACHER_CANDIDATES)].sort_values("val_macro_f1", ascending=False)
best_candidate = str(temporal_df.iloc[0]["candidate"])
print("Temporal teacher selected by validation only:", best_candidate)


In [ ]:
# ============================================================
# CELL 10 — RESTORE SELECTED TEMPORAL TEACHER + CLEAN METRICS
# ============================================================
teacher = make_model(best_candidate).to(DEVICE)
teacher.load_state_dict(candidate_states[best_candidate])
teacher.eval()

selected_train = eval_model(teacher, train_loader)
selected_val = eval_model(teacher, val_loader)
selected_test, selected_test_outputs = eval_model(teacher, test_loader, return_outputs=True)

teacher_summary = pd.DataFrame([
    {"split":"train", **selected_train},
    {"split":"val", **selected_val},
    {"split":"test", **selected_test},
])
teacher_summary.to_csv(REPORT_ROOT / "stage31_teacher_metrics.csv", index=False)
display(teacher_summary)

print("Selected temporal teacher:", best_candidate)


## 4. Scientific diagnostics for the selected teacher

The teacher must pass more than raw classification accuracy/F1.

We check:

- token-level action information;
- temporal embedding diversity;
- temporal-order sensitivity;
- temporal importance concentration;
- subject-level held-out performance;
- whether the model collapses when temporal order is destroyed.

In [ ]:
# ============================================================
# CELL 11 — TEMPORAL ORDER / TOKEN / IMPORTANCE / SUBJECT DIAGNOSTICS
# ============================================================
def temporal_shuffle_eval(model, loader, seed=SEED + 31415):
    g = torch.Generator().manual_seed(seed)
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb, _ in loader:
            perm = torch.stack([torch.randperm(SEQ_LEN, generator=g) for _ in range(len(xb))])
            xb2 = torch.gather(xb, 1, perm.unsqueeze(-1).expand(-1, -1, EEG_DIM)).to(DEVICE)
            with autocast_ctx():
                out = model(xb2)
            ys.extend(yb.numpy().tolist())
            ps.extend(out["logits"].argmax(1).cpu().numpy().tolist())
    return metrics(np.asarray(ys), np.asarray(ps))

def subject_metrics(model, loader):
    model.eval()
    store = {}
    with torch.no_grad():
        for xb, yb, idx in loader:
            xb = xb.to(DEVICE)
            with autocast_ctx():
                out = model(xb)
            pred = out["logits"].argmax(1).cpu().numpy()
            truth = yb.numpy()
            for j, ridx in enumerate(idx.numpy()):
                subject = loader.dataset.df.iloc[int(ridx)].subject_id
                store.setdefault(subject, [[], []])
                store[subject][0].append(int(truth[j]))
                store[subject][1].append(int(pred[j]))
    rows = []
    for subject, (yy, pp) in sorted(store.items()):
        rows.append({"subject_id": subject, **metrics(np.asarray(yy), np.asarray(pp))})
    return pd.DataFrame(rows)

teacher_diagnostics = {
    "selected_candidate": best_candidate,
    "chance_macro_f1": CHANCE_F1,
    "train": selected_train,
    "val": selected_val,
    "test": selected_test,
}

# Token and importance diagnostics.
teacher.eval()
all_imp = []
all_tok = []
ys_tok = []
ps_tok = []
with torch.no_grad():
    for xb, yb, _ in test_loader:
        xb = xb.to(DEVICE)
        with autocast_ctx():
            out = teacher(xb)
        token_pred = out["token_logits"].argmax(-1).cpu().numpy()
        ys_tok.extend(np.repeat(yb.numpy()[:, None], SEQ_LEN, axis=1).reshape(-1).tolist())
        ps_tok.extend(token_pred.reshape(-1).tolist())
        all_imp.append(out["importance"].float().cpu())
        all_tok.append(out["tokens"].float().cpu())

test_importance = torch.cat(all_imp, dim=0)
test_tokens = torch.cat(all_tok, dim=0)
token_probe = metrics(np.asarray(ys_tok), np.asarray(ps_tok))

entropy = -(test_importance.clamp_min(1e-8) * test_importance.clamp_min(1e-8).log()).sum(-1)
uniform_entropy = math.log(SEQ_LEN)
norm_entropy = float(entropy.mean() / uniform_entropy)

norm_tok = F.normalize(test_tokens, dim=-1)
adjacent = (norm_tok[:, 1:] * norm_tok[:, :-1]).sum(-1)
shuffle_metrics = temporal_shuffle_eval(teacher, test_loader)
shuffle_drop = float(selected_test["macro_f1"] - shuffle_metrics["macro_f1"])

# Subject-level test metrics on unseen subjects.
subject_df = subject_metrics(teacher, test_loader)
subject_df.to_csv(REPORT_ROOT / "teacher_test_subject_metrics.csv", index=False)

teacher_diagnostics.update({
    "token_probe": token_probe,
    "importance_mean_entropy": float(entropy.mean()),
    "importance_uniform_entropy": uniform_entropy,
    "importance_normalized_entropy": norm_entropy,
    "importance_mean_max": float(test_importance.max(-1).values.mean()),
    "adjacent_cosine_mean": float(adjacent.mean()),
    "adjacent_cosine_p95": float(torch.quantile(adjacent, 0.95)),
    "temporal_difference_cosine_mean": float((1.0 - adjacent).mean()),
    "temporal_shuffle_test": shuffle_metrics,
    "temporal_shuffle_macro_f1_drop": shuffle_drop,
    "subject_macro_f1_mean": float(subject_df.macro_f1.mean()),
    "subject_macro_f1_std": float(subject_df.macro_f1.std(ddof=1)),
})

with open(REPORT_ROOT / "stage31_teacher_diagnostics.json", "w") as f:
    json.dump(teacher_diagnostics, f, indent=2)

print(json.dumps(teacher_diagnostics, indent=2))
display(subject_df)


In [ ]:
# ============================================================
# CELL 12 — RANDOM-LABEL SANITY CHECK ON HELD-OUT VALIDATION
# ============================================================
# A model trained on a deterministic permutation of train labels is evaluated
# against the TRUE validation labels. This should remain near chance.
rng = np.random.default_rng(SEED + 991)
shuffled_train_labels = train_seq["label_id"].to_numpy()[rng.permutation(len(train_seq))]

class ShuffledLabelDataset(Dataset):
    def __init__(self, base, shuffled_labels):
        self.base = base
        self.labels = shuffled_labels.astype(np.int64)
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        x, _, idx = self.base[i]
        return x, int(self.labels[i]), idx

san_ds = ShuffledLabelDataset(train_ds, shuffled_train_labels)
san_loader = DataLoader(san_ds, shuffle=True, **loader_kwargs)
sanity = MeanStdMLP().to(DEVICE)
san_opt = torch.optim.AdamW(sanity.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

for _ in range(2):
    sanity.train()
    for xb, yb, _ in san_loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE)
        san_opt.zero_grad(set_to_none=True)
        with autocast_ctx():
            loss = candidate_loss("mean_mlp", sanity(xb), yb)
        loss.backward()
        san_opt.step()

# Evaluate the random-label-trained model on the real validation labels.
random_val = eval_model(sanity, val_loader)
with open(REPORT_ROOT / "random_label_sanity.json", "w") as f:
    json.dump(random_val, f, indent=2)
print("Random-label validation result:", random_val)


## 5. EEG-teacher stage gate

The gate is conservative and was defined before seeing the final test metrics.

A Phase-3.1 **PASS** requires the temporal teacher to be meaningfully above chance **and** to exhibit evidence that the signal is genuinely temporal rather than a static or collapsed shortcut.

A gate failure/inconclusive result means **do not proceed to neural-to-visual transfer**.

In [ ]:
# ============================================================
# CELL 14 — SAVE CANONICAL TEACHER CHECKPOINT + CACHE NEURAL TARGETS
# ============================================================
teacher_ckpt = MODEL_ROOT / "stage31_eeg_teacher_best.pt"

# Save the best temporal teacher regardless of gate status so the run is auditable.
status = "EXPERIMENTAL EVIDENCE COMPLETE — TEMPORAL SIGNAL WEAK"
torch.save({
    "model_state": teacher.state_dict(),
    "architecture": best_candidate,
    "extra": {
        "phase": "3.1",
        "sequence_length": SEQ_LEN,
        "eeg_dim": EEG_DIM,
        "num_classes": NUM_CLASSES,
        "seed": SEED,
        "gate_status": status,
        "best_val_macro_f1": float(teacher_metrics["val"]["macro_f1"]),
        "token_loss_weight": TOKEN_LOSS_WEIGHT,
        "importance_temperature": IMPORTANCE_TEMPERATURE,
    },
}, teacher_ckpt)

# Cache token geometry + temporal importance only for a temporal teacher.
def cache_split(table, loader, split_name):
    teacher.eval()
    toks, imps, ids = [], [], []
    with torch.no_grad():
        for xb, _, idx in loader:
            xb = xb.to(DEVICE)
            with autocast_ctx():
                out = teacher(xb)
            toks.append(out["tokens"].float().cpu())
            imps.append(out["importance"].float().cpu())
            ids.append(idx.numpy())
    ids = np.concatenate(ids)
    order = np.argsort(ids)
    tok = torch.cat(toks, 0)[order]
    imp = torch.cat(imps, 0)[order]
    if CACHE_FP16:
        tok = tok.half(); imp = imp.half()
    torch.save(tok, CACHE_ROOT / f"{split_name}_teacher_tokens.pt")
    torch.save(imp, CACHE_ROOT / f"{split_name}_teacher_importance.pt")
    print(split_name, tuple(tok.shape), tuple(imp.shape))

cache_split(train_seq, train_loader, "train")
cache_split(val_seq, val_loader, "val")
cache_split(test_seq, test_loader, "test")

torch.save(train_eeg_mean, OUT_ROOT / "train_eeg_mean.pt")
torch.save(train_eeg_std, OUT_ROOT / "train_eeg_std.pt")

print("Canonical teacher checkpoint:", teacher_ckpt)
print("Gate status:", status)


In [ ]:
# ============================================================
# CELL 15 — RESEARCH-GRADE FIGURES
# ============================================================
plt.figure(figsize=(9, 5))
plot_df = candidate_df.sort_values("val_macro_f1", ascending=True)
plt.barh(plot_df["candidate"], plot_df["val_macro_f1"])
plt.axvline(CHANCE_F1, linestyle="--", label="27-class chance")
plt.xlabel("Validation Macro-F1")
plt.ylabel("Candidate")
plt.title("EEG teacher candidate comparison")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_ROOT / "01_candidate_validation.png", dpi=200)
plt.close()

plt.figure(figsize=(8, 5))
plt.hist(adjacent.numpy().reshape(-1), bins=50)
plt.axvline(ADJACENT_COSINE_MAX, linestyle="--", label="collapse threshold")
plt.xlabel("Adjacent-token cosine similarity")
plt.ylabel("Count")
plt.title("EEG temporal representation diversity")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_ROOT / "02_adjacent_cosine_histogram.png", dpi=200)
plt.close()

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, SEQ_LEN + 1), test_importance.mean(0).numpy(), marker="o")
plt.xlabel("Second in 8-second sequence")
plt.ylabel("Mean EEG temporal importance")
plt.xticks(np.arange(1, SEQ_LEN + 1))
plt.title("EEG teacher temporal importance")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(FIG_ROOT / "03_temporal_importance.png", dpi=200)
plt.close()

plt.figure(figsize=(10, 5))
train_dist = seq_df[seq_df.split == "train"]["label_id"].value_counts().sort_index()
plt.bar(train_dist.index, train_dist.values)
plt.xlabel("Action class ID")
plt.ylabel("Training sequences")
plt.title("Frozen Phase-2 training class distribution")
plt.tight_layout()
plt.savefig(FIG_ROOT / "04_class_distribution.png", dpi=200)
plt.close()

plt.figure(figsize=(8, 5))
plt.bar(subject_df["subject_id"], subject_df["macro_f1"])
plt.xlabel("Held-out test subject")
plt.ylabel("Macro-F1")
plt.title("EEG teacher subject-level test Macro-F1")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_ROOT / "05_subject_macro_f1.png", dpi=200)
plt.close()

print("Figures saved to:", FIG_ROOT)


In [ ]:
# ============================================================
# CELL 16 — PACKAGE PHASE 3.1 ARTIFACTS
# ============================================================
PACKAGE_BASE = OUT_ROOT.parent / "egobrain_phase3_1_eeg_teacher_artifacts"
ZIP_PATH = OUT_ROOT.parent / "egobrain_phase3_1_eeg_teacher_artifacts.zip"

if PACKAGE_BASE.exists():
    shutil.rmtree(PACKAGE_BASE)
PACKAGE_BASE.mkdir(parents=True)

for src in [MODEL_ROOT, REPORT_ROOT, FIG_ROOT, CACHE_ROOT]:
    if src.exists():
        shutil.copytree(src, PACKAGE_BASE / src.name)

for src in [OUT_ROOT / "train_eeg_mean.pt", OUT_ROOT / "train_eeg_std.pt"]:
    if src.exists():
        shutil.copy2(src, PACKAGE_BASE / src.name)

manifest = {
    "phase": "phase_3_1_eeg_teacher",
    "seed": SEED,
    "device": str(DEVICE),
    "selected_temporal_teacher": best_candidate,
    "gate_status": status,
    "sequence_length": SEQ_LEN,
    "eeg_dim": EEG_DIM,
    "num_classes": NUM_CLASSES,
    "phase1_labeled_index": str(LABELED_INDEX),
    "phase2_split": str(PHASE2_SPLIT),
    "eeg_root": str(EEG_ROOT),
    "train_subjects": train_subjects,
    "val_subjects": val_subjects,
    "test_subjects": test_subjects,
    "required_next_step": "Proceed to Phase 3.2 only if phase31 gate status is PASS.",
}
with open(PACKAGE_BASE / "phase31_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

readme = f"""EgoBrain Phase 3.1 — EEG Temporal Teacher\n\nGate status: {status}\nSelected temporal teacher: {best_candidate}\n\nUse this package for Phase 3.2 only when the gate is PASS.\nThe package contains the teacher checkpoint, train-only normalization statistics, cached neural tokens/importance, reports, figures and checksums.\n"""
(PACKAGE_BASE / "README.txt").write_text(readme)

hash_rows = []
for p in sorted(PACKAGE_BASE.rglob("*")):
    if p.is_file():
        h = hashlib.sha256()
        with p.open("rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)
        hash_rows.append({
            "relative_path": str(p.relative_to(PACKAGE_BASE)),
            "sha256": h.hexdigest(),
            "bytes": int(p.stat().st_size),
        })
pd.DataFrame(hash_rows).to_csv(PACKAGE_BASE / "artifact_sha256.csv", index=False)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", root_dir=PACKAGE_BASE)

print("=" * 78)
print("PHASE 3.1 ARTIFACT PACKAGE")
print("=" * 78)
print("STATUS:", status)
print("Teacher checkpoint:", teacher_ckpt)
print("Artifact ZIP:", ZIP_PATH)
print("ZIP MB:", round(ZIP_PATH.stat().st_size / 1024**2, 2))


# EEG-teacher stage hand-off

### PASS path
If the final gate says:

```text
EEG-TEACHER STAGE CLEARED — freeze this teacher and proceed to Phase 3.2.
```

then persist the complete artifact package before starting a new Kaggle Commit:

```text
/kaggle/working/egobrain_phase3_1_eeg_teacher_artifacts.zip
```

and/or save the folder as a Kaggle Dataset/Input.

### INCONCLUSIVE path
Do **not** start Phase 3.2 neural transfer. Use the reports to determine whether the bottleneck is:

- EEG feature collapse;
- lack of EEG action information;
- weak temporal order dependence;
- nearly uniform teacher importance;
- class imbalance / optimization;
- or a mismatch between the intended neural signal and the 1-second feature representation.

### Important
EEG-teacher stage passing is a **prerequisite**, not automatic completion of the full NeuroRoute phase. The later stages (3.2–3.8) must still independently pass their scientific checks before the eventual Phase-4 hand-off.